In [ ]:
# Installing libraries
!pip install transformers
!pip install datasets==2.19.1
!pip install seqeval
!pip install evaluate

In [ ]:
# Importing libraries
from datasets import load_dataset

from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from transformers import TrainingArguments
from transformers import Trainer
from transformers import DataCollatorForTokenClassification

import evaluate

import numpy as np

In [ ]:
# Loading Dataset
from datasets import load_dataset

dataset = load_dataset("conll2003")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [ ]:
small_train = dataset["train"].shuffle(seed=42).select(range(1000))

small_val = dataset["validation"].shuffle(seed=42).select(range(200))

small_test = dataset["test"].shuffle(seed=42).select(range(200))

dataset["train"]=small_train

dataset["validation"]=small_val

dataset["test"]=small_test

dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 200
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 200
    })
})

In [ ]:
dataset["train"][0]

{'id': '1469',
 'tokens': ['"',
  'Neither',
  'the',
  'National',
  'Socialists',
  '(',
  'Nazis',
  ')',
  'nor',
  'the',
  'communists',
  'dared',
  'to',
  'kidnap',
  'an',
  'American',
  'citizen',
  ',',
  '"',
  'he',
  'shouted',
  ',',
  'in',
  'an',
  'oblique',
  'reference',
  'to',
  'his',
  'extradition',
  'to',
  'Germany',
  'from',
  'Denmark',
  '.',
  '"'],
 'pos_tags': [0,
  12,
  12,
  22,
  23,
  4,
  23,
  5,
  10,
  12,
  24,
  38,
  35,
  37,
  12,
  16,
  21,
  6,
  0,
  28,
  38,
  6,
  15,
  12,
  16,
  21,
  35,
  29,
  21,
  35,
  22,
  15,
  22,
  7,
  0],
 'chunk_tags': [0,
  11,
  11,
  12,
  12,
  0,
  11,
  0,
  0,
  11,
  12,
  21,
  22,
  22,
  11,
  12,
  12,
  0,
  0,
  11,
  21,
  0,
  13,
  11,
  12,
  12,
  13,
  11,
  12,
  13,
  11,
  13,
  11,
  12,
  0],
 'ner_tags': [0,
  0,
  0,
  7,
  8,
  0,
  7,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  7,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  5,
  0,
  5,
  0

In [ ]:
# Labels
pos_labels = dataset["train"].features["pos_tags"].feature.names

chunk_labels = dataset["train"].features["chunk_tags"].feature.names

print("POS Labels:",pos_labels)

print("Chunk Labels:",chunk_labels)

POS Labels: ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
Chunk Labels: ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(

"distilbert-base-uncased"

)

In [ ]:
# Token alignment
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(

        examples["tokens"],

        truncation=True,

        padding='max_length',

        max_length=64,

        is_split_into_words=True

    )

    labels=[]

    for i,label in enumerate(examples["pos_tags"]):

        word_ids=tokenized_inputs.word_ids(batch_index=i)

        previous_word=None

        label_ids=[]

        for word_idx in word_ids:

            if word_idx is None:

                label_ids.append(-100)

            elif word_idx!=previous_word:

                label_ids.append(label[word_idx])

            else:

                label_ids.append(-100)

            previous_word=word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"]=labels

    return tokenized_inputs

In [ ]:
# Apply preprocessing
tokenized_dataset = dataset.map(

tokenize_and_align_labels,

batched=True
)

tokenized_dataset

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
})

In [ ]:
# Model
model = AutoModelForTokenClassification.from_pretrained(

"distilbert-base-uncased",

num_labels=len(pos_labels)

)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Training Setup
training_args = TrainingArguments(

output_dir="results",

learning_rate=2e-5,

per_device_train_batch_size=8,

per_device_eval_batch_size=8,

num_train_epochs=1,

weight_decay=0.01

)

In [ ]:
# data_collator
data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
# trainer
trainer = Trainer(

model=model,

args=training_args,

train_dataset=tokenized_dataset["train"],

eval_dataset=tokenized_dataset["validation"],

data_collator=data_collator

)

In [ ]:
#Training
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=125, training_loss=2.372457275390625, metrics={'train_runtime': 353.5377, 'train_samples_per_second': 2.829, 'train_steps_per_second': 0.354, 'total_flos': 16344925824000.0, 'train_loss': 2.372457275390625, 'epoch': 1.0})

In [ ]:
#Evauation
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 1.7605013847351074,
 'eval_runtime': 25.8023,
 'eval_samples_per_second': 7.751,
 'eval_steps_per_second': 0.969,
 'epoch': 1.0}

In [ ]:
# Metrics
metric = evaluate.load("seqeval")

predictions = trainer.predict(

    tokenized_dataset["validation"]

)

preds = np.argmax(

    predictions.predictions,

    axis=2

)

true_labels = predictions.label_ids


# convert ids to label names and remove -100
true_predictions = []
true_references = []

for pred,label in zip(preds,true_labels):

    temp_pred=[]
    temp_label=[]

    for p,l in zip(pred,label):

        if l != -100:

            temp_pred.append(pos_labels[p])

            temp_label.append(pos_labels[l])

    true_predictions.append(temp_pred)

    true_references.append(temp_label)


results = metric.compute(

    predictions=true_predictions,

    references=true_references

)

print("Precision:",results["overall_precision"])

print("Recall:",results["overall_recall"])

print("F1:",results["overall_f1"])

print("Accuracy:",results["overall_accuracy"])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Precision: 0.5128348214285714
Recall: 0.3620961386918834
F1: 0.42448036951501156
Accuracy: 0.5553410553410554


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: JJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNS seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VBP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserW

In [ ]:
#Inference

sentence = "John works at Google in California"

tokens = tokenizer(

    sentence.split(),

    is_split_into_words=True,

    return_tensors="pt",

    padding=True,

    truncation=True

)

outputs = model(**tokens)

preds = np.argmax(

    outputs.logits.detach().numpy(),

    axis=2

)

predicted_labels = [

    pos_labels[p]

    for p in preds[0][:len(sentence.split())]

]

print("Sentence:",sentence)

print("Predicted POS tags:",predicted_labels)

Sentence: John works at Google in California
Predicted POS tags: ['NNP', 'NNP', 'NNP', 'IN', 'NNP', 'IN']


## POS Tagging vs Chunking Comparison

POS Tagging:
POS tagging assigns grammatical categories to each word such as noun, verb, adjective etc.

Example:
John → Noun
works → Verb

Chunking:
Chunking groups words into meaningful phrases such as noun phrases or verb phrases.

Example:
John works → Noun Phrase

Key Differences:

POS tagging:
• Word level classification
• Identifies grammar roles

Chunking:
• Phrase level classification
• Identifies sentence structure

Conclusion:

Chunking is more complex because it considers relationships between words.


## TFinal Report

### Differences between POS Tagging and Chunking

Part-of-Speech (POS) tagging and Chunking are both important Natural Language Processing tasks used for understanding the structure of text.

POS tagging assigns a grammatical category to each individual word in a sentence such as noun, verb, adjective, or preposition. It focuses on identifying the role of each word independently.

Example:
Sentence: John works at Google  
POS Tags:
John → Noun  
works → Verb  
at → Preposition  
Google → Proper Noun  

Chunking, also known as shallow parsing, groups words together into meaningful phrases such as noun phrases (NP) or verb phrases (VP). Instead of focusing on individual words, chunking focuses on phrase structure.

Example:
John works → Noun Phrase  
at Google → Prepositional Phrase  

Key Difference:

POS tagging works at the word level, while chunking works at the phrase level. Chunking is slightly more complex because it requires understanding relationships between words.

---

### Challenges Faced

During this assignment several challenges were encountered while implementing token classification using DistilBERT.

One major challenge was aligning POS labels with tokenized words because transformer models split words into subwords. This required careful label alignment so that correct labels were assigned.

Another challenge was handling padding tokens because they should not affect model evaluation. This required ignoring padding labels using special values like -100.

We also faced issues related to library compatibility such as changes in HuggingFace APIs and metric loading functions, which required version adjustments.

Training time was another challenge, especially when running on CPU. This was handled by reducing dataset size while maintaining the correctness of the implementation.

---

### Observations and Insights

From this task, it was observed that transformer models like DistilBERT perform very well for sequence labeling tasks because they capture contextual relationships between words.

Fine tuning pretrained transformer models significantly improves performance compared to training models from scratch.

Another important insight was that proper preprocessing and label alignment are critical steps in token classification tasks.

This task also demonstrated how HuggingFace Trainer API simplifies model training and evaluation.

---

### Conclusion

This assignment helped in understanding how transformer models can be applied for token classification tasks such as POS tagging and chunking. It also provided practical experience in preprocessing, training, evaluation, and inference using modern NLP tools.